In [ ]:
from pathlib import Path
import sys
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

current_dir = Path.cwd().resolve()
project_root = current_dir.parent if current_dir.name == 'notebooks' else current_dir
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.preprocess import load_and_preprocess_data
from src.logistic_scratch import LogisticRegressionScratchOvR
from src.evaluate import evaluate_model, plot_confusion_matrices

# Use the same data split and model settings as main.py.
data_path = project_root / 'data' / 'iris_extended.csv'
df, X_train, X_test, y_train, y_test, encoder = load_and_preprocess_data(data_path)

scratch_model = LogisticRegressionScratchOvR(lr=0.1, num_iter=1000)
scratch_model.fit(X_train, y_train)
scratch_preds = scratch_model.predict(X_test)
scratch_metrics, scratch_cm = evaluate_model(
    y_test, scratch_preds, 'Logistic Regression (Scratch)'
)

rf_model = RandomForestClassifier(n_estimators=100, random_state=21)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)
rf_metrics, rf_cm = evaluate_model(y_test, rf_preds, 'Random Forest (Library)')

results = pd.DataFrame([scratch_metrics, rf_metrics])
display(results)

figure_path = project_root / 'outputs' / 'figures' / 'confusion_matrices.png'
plot_confusion_matrices(
    scratch_cm,
    rf_cm,
    'Logistic Regression (Scratch)',
    'Random Forest (Library)',
    encoder.classes_,
    save_path=str(figure_path),
)
